In [22]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.notebook import tqdm
import seaborn as sns

from tensorflow.keras.models import load_model
import matplotlib as plt

##%%
# Add project root so we can import utils
project_root = Path().cwd().parent.resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Imports from src.utils (file_paths is used by the utility modules)
from src.utils.file_paths import file_paths 

# Imports from our new testing_utils location: src/utils/testing_utils/
from src.utils.testing_utils.data_utils import get_sample_sets, load_sample_files, merge_and_prepare_data
from src.utils.testing_utils.model_utils import load_and_predict as load_pr_model_and_predict
from src.utils.testing_utils.model_utils import compute_classification_metrics
from src.utils.testing_utils.mueller_utils import generate_method_a_matrix, generate_method_b_matrix, generate_method_c_matrix
from src.utils.testing_utils.comparison_utils import compute_direct_comparisons, analyze_comparison_results
from src.utils.testing_utils.visualization_utils import save_visualizations
from src.utils.testing_utils.decomposition_utils import analyze_lu_chipman_decomposition
from src.utils.testing_utils.direct_filtering_utils import compare_filtering_methods, plot_roc_curve, plot_precision_recall_curve

s## Core Workflow Functions (using utilities)
# These functions orchestrate the calls to the `testing_utils` modules.

In [2]:
def process_single_sample(arr, sample_name, last_row_gen_model, ml_pr_mask_flat, tissue_type=None, visualize=False):
    """
    Process a single sample: generate Mueller matrices for methods A, B, C.
    Uses utility functions for MM generation and visualization.

    Args:
        arr (np.ndarray): Sample array (H, W, 17).
        sample_name (str): Name of the sample.
        last_row_gen_model (tf.keras.Model): Loaded Keras model for last row generation.
        ml_pr_mask_flat (np.ndarray): Flattened ML-predicted PR mask for this sample.
        tissue_type (str, optional): Type of tissue (for visualization path). Defaults to None.
        visualize (bool, optional): Whether to save visualizations. Defaults to False.

    Returns:
        dict: Results including generated matrices, SSIM comparisons, and PR indices.
    """
    H, W, _ = arr.shape
    n_pix = H * W

    # Method A: Original PR with original last row
    method_a_matrix, pr_indices = generate_method_a_matrix(arr, H, W)

    # Method B: Original PR with generated last row
    method_b_matrix = generate_method_b_matrix(arr, H, W, last_row_gen_model)

    # Method C: ML PR with generated last row
    method_c_matrix, ml_pr_indices = generate_method_c_matrix(
        arr, H, W, last_row_gen_model, ml_pr_mask_flat
    )

    print(f"\nProcessed Sample {sample_name}:")
    print(f"  Original PR pixels: {len(pr_indices)} ({len(pr_indices)/n_pix*100:.2f}%)")
    print(f"  ML PR pixels: {len(ml_pr_indices)} ({len(ml_pr_indices)/n_pix*100:.2f}%)")

    # Compute SSIM comparisons of the full Mueller Matrices
    a_vs_b_ssim, a_vs_c_ssim, b_vs_c_ssim = compute_direct_comparisons(
        method_a_matrix, method_b_matrix, method_c_matrix, sample_name
    )

    # Apply M[0,0] normalization if tissue is cervix
    if tissue_type == 'cervix':
        print(f"DEBUG: Applying cervix M[0,0] normalization for {sample_name}")
        for i, matrix in enumerate((method_a_matrix, method_b_matrix, method_c_matrix)):
            if matrix is None:
                continue

            m00 = matrix[..., 0, 0]
            non_zero = m00 != 0

            # Print debug info only once for the first matrix
            if i == 0 and not getattr(process_single_sample, "_cervix_debug_done", False):
                flat = m00.flatten()
                print("DEBUG: M[0,0] before normalization (first 10):", flat[:10])
                print("DEBUG: unique values:", np.unique(flat[flat != 0]))
                process_single_sample._cervix_debug_done = True

            matrix[..., 0, 0][non_zero] = 1

    # Save visualizations if requested
    if visualize and tissue_type:
        flat_a = method_a_matrix.reshape(n_pix, 16)
        flat_b = method_b_matrix.reshape(n_pix, 16)
        flat_c = method_c_matrix.reshape(n_pix, 16)
        save_visualizations(flat_a, flat_b, flat_c, H, W, sample_name, tissue_type)
        print(f"Visualizations saved for {sample_name}")

    return {
        'sample_name': sample_name,
        'method_a_matrix': method_a_matrix,
        'method_b_matrix': method_b_matrix,
        'method_c_matrix': method_c_matrix,
        'direct_mm_ssim': {
            'a_vs_b': a_vs_b_ssim,
            'a_vs_c': a_vs_c_ssim,
            'b_vs_c': b_vs_c_ssim,
        },
        'pr_indices': pr_indices,
        'ml_pr_indices': ml_pr_indices
    }

In [3]:
def evaluate_tissue_samples(tissue_type, sample_filenames_set, pr_model_name="xgb", visualize_first_n=1):
    """
    Run evaluation workflow for a specific tissue type and PR model.
    1. Loads data.
    2. Performs PR prediction using the specified ML model.
    3. Computes PR classification metrics.
    4. Loads the last-row generation model.
    5. Processes each sample to generate Mueller matrices (Methods A, B, C).
    6. Aggregates and saves direct Mueller Matrix SSIM comparison results.

    Args:
        tissue_type (str): E.g., 'brain', 'cervix'.
        sample_filenames_set (set): Set of sample filenames to load for this tissue.
        pr_model_name (str): PR model to use ('mlp', 'xgb', or 'cat').
        visualize_first_n (int): Number of initial samples to visualize MM images for.

    Returns:
        dict: Aggregated evaluation results for the tissue type, or None if errors occur.
              This dict is the `results_eval` expected by `analyze_lu_chipman_decomposition`.
    """
    print(f"\\n{'='*20} Evaluating {tissue_type.upper()} samples (PR Model: {pr_model_name.upper()}) {'='*20}")

    # 1. Load sample files using data_utils
    arrays, sample_names, dims = load_sample_files(tissue_type, sample_filenames_set)
    if not arrays:
        print(f"No data loaded for {tissue_type}. Aborting evaluation.")
        return None

    # 2. Prepare data using data_utils
    X_flat, y_flat_gt_pr, iso_merged = merge_and_prepare_data(arrays)
    if X_flat is None:
        print(f"Data preparation failed for {tissue_type}. Aborting evaluation.")
        return None

    # 3. Load PR model and predict using model_utils
    y_pred_ml_pr, probs_ml_pr = load_pr_model_and_predict(X_flat, use_model=pr_model_name)

    # 4. Compute PR classification metrics using model_utils
    pr_classification_metrics = compute_classification_metrics(y_flat_gt_pr, y_pred_ml_pr, probs_ml_pr)

    # 5. Load last-row generation model (nn_model_v2b.keras)
    try:
        last_row_gen_model_path = file_paths.model_save_path / 'nn_model_v2b.keras'
        last_row_gen_model = load_model(last_row_gen_model_path)
        print(f"Last-row generation model loaded: {last_row_gen_model_path}")
    except Exception as e:
        print(f"CRITICAL ERROR: Failed to load last-row generation model: {e}")
        return None

    # 6. Reshape ML PR predictions back to image dimensions for per-sample processing
    num_samples, H, W, _ = iso_merged.shape
    ml_pr_mask_per_sample = y_pred_ml_pr.reshape(num_samples, H, W) # (ns, h, w)

    # 7. Process each sample
    all_sample_processing_results = []
    desc = f"Processing {tissue_type} samples ({pr_model_name} PR)"
    for idx in tqdm(range(len(arrays)), desc=desc, leave=False):
        current_sample_array = iso_merged[idx] # (H, W, 17)
        current_sample_name = sample_names[idx]
        current_ml_pr_mask_flat = ml_pr_mask_per_sample[idx].reshape(-1) # Flattened (H*W,)

        should_visualize = (idx < visualize_first_n)

        single_sample_res = process_single_sample(
            current_sample_array, current_sample_name, last_row_gen_model,
            current_ml_pr_mask_flat, tissue_type=tissue_type, visualize=should_visualize
        )
        all_sample_processing_results.append(single_sample_res)

    # 8. Aggregate results for direct MM SSIM
    # These are lists of (H,W,4,4) np.ndarray
    method_a_matrices_list = [res['method_a_matrix'] for res in all_sample_processing_results]
    method_b_matrices_list = [res['method_b_matrix'] for res in all_sample_processing_results]
    method_c_matrices_list = [res['method_c_matrix'] for res in all_sample_processing_results]

    # These are lists of 1D np.ndarray (flattened indices)
    gt_pr_indices_list = [res['pr_indices'] for res in all_sample_processing_results]
    ml_pr_indices_list = [res['ml_pr_indices'] for res in all_sample_processing_results]

    # DataFrames for direct MM SSIM results (each row is a sample, cols are M_ij SSIMs)
    df_a_vs_b_mm_ssim = pd.DataFrame([res['direct_mm_ssim']['a_vs_b'] for res in all_sample_processing_results])
    df_a_vs_c_mm_ssim = pd.DataFrame([res['direct_mm_ssim']['a_vs_c'] for res in all_sample_processing_results])
    df_b_vs_c_mm_ssim = pd.DataFrame([res['direct_mm_ssim']['b_vs_c'] for res in all_sample_processing_results])

    # Save raw MM SSIM DataFrames
    # Output path: results/{tissue_type}/direct_mm_ssim/{pr_model_name}/
    eval_out_dir_mm_ssim = file_paths.results / tissue_type / "direct_mm_ssim" / pr_model_name
    eval_out_dir_mm_ssim.mkdir(exist_ok=True, parents=True)
    if not df_a_vs_b_mm_ssim.empty: df_a_vs_b_mm_ssim.to_csv(eval_out_dir_mm_ssim / f'a_vs_b_mm_ssim.csv', index=False)
    if not df_a_vs_c_mm_ssim.empty: df_a_vs_c_mm_ssim.to_csv(eval_out_dir_mm_ssim / f'a_vs_c_mm_ssim.csv', index=False)
    if not df_b_vs_c_mm_ssim.empty: df_b_vs_c_mm_ssim.to_csv(eval_out_dir_mm_ssim / f'b_vs_c_mm_ssim.csv', index=False)

    # Analyze and summarize direct MM SSIM results using comparison_utils
    # This saves its own summary CSV and plot to results/{tissue_type}/
    avg_direct_mm_ssim_summary_df = analyze_comparison_results(
        df_a_vs_b_mm_ssim, df_a_vs_c_mm_ssim, df_b_vs_c_mm_ssim,
        tissue_type, pr_model_name
    )

    print(f"\\n{tissue_type.upper()} evaluation complete. Direct MM SSIM results/summaries saved under {file_paths.results / tissue_type}.")

    # This dictionary structure is crucial for `analyze_lu_chipman_decomposition`
    return {
        'tissue_type': tissue_type,
        'pr_model_name': pr_model_name, # For tagging outputs
        'pr_classification_metrics': pr_classification_metrics,
        'avg_direct_mm_ssim_summary': avg_direct_mm_ssim_summary_df, # DataFrame summarizing MM element SSIMs
        'matrices': { # Crucial for decomposition: lists of (H,W,4,4) matrices
            'method_a': method_a_matrices_list,
            'method_b': method_b_matrices_list,
            'method_c': method_c_matrices_list
        },
        # The following are useful if decomposition needs to mask visualizations by PR region
        'gt_pr_indices_list': gt_pr_indices_list,
        'ml_pr_indices_list': ml_pr_indices_list,
        'sample_names': sample_names, # Corresponding sample names, useful for decomposition logs
        # Raw SSIM data for MMs, can be useful for detailed inspection
        'comparison_results': { # This matches a key used by original decomp function to get sample names
            'a_vs_b': df_a_vs_b_mm_ssim # Sample names can be derived from the 'Sample' column here
        }
    }


In [5]:
def create_overall_summary(all_tissues_eval_results, overall_pr_model_name):
    """
    Create summary comparisons across all tissue types from their evaluation results.
    Focuses on:
    1. PR Classification Metrics.
    2. Average Direct Mueller Matrix SSIMs (element-wise averages).
    
    Args:
        all_tissues_eval_results (dict): Dict where keys are tissue_type and values are outputs from evaluate_tissue_samples.
        overall_pr_model_name (str): The PR model name used for these evaluations (for naming summary files).
    """
    if not all_tissues_eval_results:
        print("No evaluation results provided to create an overall summary.")
        return
    
    # Define a specific subdirectory for these overall summaries
    summary_output_dir = file_paths.results / 'overall_summary' / overall_pr_model_name
    summary_output_dir.mkdir(exist_ok=True, parents=True)
    
    # 1. PR Classification Metrics Summary
    pr_metrics_list_for_summary = []
    for tissue, eval_res_data in all_tissues_eval_results.items():
        if eval_res_data and 'pr_classification_metrics' in eval_res_data:
            metrics = eval_res_data['pr_classification_metrics'].copy() # Avoid modifying original
            metrics['tissue'] = tissue 
            pr_metrics_list_for_summary.append(metrics)
    
    if pr_metrics_list_for_summary:
        df_pr_class_summary = pd.DataFrame(pr_metrics_list_for_summary).set_index('tissue')
        df_pr_class_summary.to_csv(summary_output_dir / f'summary_pr_classification_metrics.csv')
        print("\\n===== Overall PR Classification Metrics Summary =====")
        print(df_pr_class_summary)
        try:
            df_pr_class_summary.plot(kind='bar', figsize=(12, 7), rot=0)
            plt.title(f'Overall PR Classification Metrics (PR Model: {overall_pr_model_name.upper()})')
            plt.ylabel('Score')
            plt.xlabel('Tissue Type')
            plt.legend(title='Metric', bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.tight_layout()
            plt.savefig(summary_output_dir / f'summary_pr_classification_metrics.png', dpi=300)
            plt.close()
        except Exception as e:
            print(f"Error plotting overall PR classification summary: {e}")
    else:
        print("No PR classification metrics found to summarize across tissues.")

    # 2. Average Direct Mueller Matrix SSIM Summary (summarizing the tissue-level summaries)
    avg_mm_ssim_list_for_summary = []
    for tissue, eval_res_data in all_tissues_eval_results.items():
        if eval_res_data and 'avg_direct_mm_ssim_summary' in eval_res_data:
            # eval_res_data['avg_direct_mm_ssim_summary'] is a DataFrame:
            # Index=M_ij, Columns=['A_vs_B', 'A_vs_C', 'B_vs_C']
            tissue_summary_df = eval_res_data['avg_direct_mm_ssim_summary']
            if not tissue_summary_df.empty:
                avg_mm_ssim_list_for_summary.append({
                    'tissue': tissue,
                    'A_vs_B_MM_SSIM_AvgElements': tissue_summary_df['A_vs_B'].mean(skipna=True),
                    'A_vs_C_MM_SSIM_AvgElements': tissue_summary_df['A_vs_C'].mean(skipna=True),
                    'B_vs_C_MM_SSIM_AvgElements': tissue_summary_df['B_vs_C'].mean(skipna=True)
                })
                
    if avg_mm_ssim_list_for_summary:
        df_avg_mm_ssim_summary = pd.DataFrame(avg_mm_ssim_list_for_summary).set_index('tissue')
        df_avg_mm_ssim_summary.to_csv(summary_output_dir / f'summary_avg_direct_mm_ssim.csv')
        print("\\n===== Overall Average Direct Mueller Matrix SSIM Summary (across MM elements) =====")
        print(df_avg_mm_ssim_summary)
        try:
            df_avg_mm_ssim_summary.plot(kind='bar', figsize=(10, 6), rot=0)
            plt.title(f'Overall Avg. Direct MM SSIM (PR Model: {overall_pr_model_name.upper()})')
            plt.ylabel('Avg. SSIM over MM elements')
            plt.xlabel('Tissue Type')
            plt.legend(title='Comparison Type')
            plt.tight_layout()
            plt.savefig(summary_output_dir / f'summary_avg_direct_mm_ssim.png', dpi=300)
            plt.close()
        except Exception as e:
            print(f"Error plotting overall avg direct MM SSIM summary: {e}")
    else:
        print("No direct MM SSIM summaries found to summarize across tissues.")

In [6]:
def run_all_tissue_evaluations(pr_model_to_use='xgb', visualize_first_n_samples_per_tissue=0):
    """
    Runs `evaluate_tissue_samples` for all defined tissue types using a specific PR model.
    Then creates an overall summary of these evaluations.
    """
    all_sample_sets = get_sample_sets() # From data_utils
    
    collected_eval_results = {}
    print(f"\\n{'*'*30} RUNNING ALL TISSUE EVALUATIONS (PR Model: {pr_model_to_use.upper()}) {'*'*30}")
    
    for tissue, samples_set in all_sample_sets.items():
        eval_res = evaluate_tissue_samples(
            tissue_type=tissue, 
            sample_filenames_set=samples_set, 
            pr_model_name=pr_model_to_use,
            visualize_first_n=visualize_first_n_samples_per_tissue
        )
        if eval_res: # Ensure evaluation was successful
            collected_eval_results[tissue] = eval_res
            
    if collected_eval_results: # Check if any evaluation succeeded
        print(f"\\n{'*'*30} CREATING OVERALL SUMMARY (PR Model: {pr_model_to_use.upper()}) {'*'*30}")
        create_overall_summary(collected_eval_results, overall_pr_model_name=pr_model_to_use)
    else:
        print("\\nNo tissue evaluations were successful. Skipping overall summary creation.")
        
    return collected_eval_results # Return all data for further use, e.g., decomposition

In [7]:
def compare_pr_models_on_tissue(tissue_type_to_test='brain', sample_set_for_comparison=None, pr_models_list=['mlp', 'xgb', 'cat']):
    """
    Compares multiple PR models on a single specified tissue type.
    Generates comparison plots for PR classification metrics and avg. direct MM SSIMs.
    """
    if sample_set_for_comparison is None:
        sample_set_for_comparison = get_sample_sets().get(tissue_type_to_test)
        if sample_set_for_comparison is None:
            print(f"ERROR: Default sample set not found for tissue '{tissue_type_to_test}'. Cannot compare PR models.")
            return None # Return None if samples not found
            
    print(f"\\n{'='*20} COMPARING PR MODELS on {tissue_type_to_test.upper()} samples {'='*20}")
    
    results_per_pr_model = {} # Store full eval output for each PR model
    for pr_model in pr_models_list:
        print(f"\\n--- Evaluating {tissue_type_to_test.upper()} with PR Model: {pr_model.upper()} ---")
        eval_res = evaluate_tissue_samples(
            tissue_type_to_test, 
            sample_set_for_comparison, 
            pr_model_name=pr_model,
            visualize_first_n=0 # No sample-level MM visualizations during model comparison
        )
        if eval_res: # Check if evaluation succeeded
            results_per_pr_model[pr_model] = eval_res

    if not results_per_pr_model: # Check if any model was successfully evaluated
        print(f"No PR models could be successfully evaluated for {tissue_type_to_test}. Comparison aborted.")
        return None

    # --- Summarize and Plot PR Model Comparison ---
    # Output path: results/pr_model_comparison/{tissue_type}/
    comp_out_dir = file_paths.results / 'pr_model_comparison' / tissue_type_to_test
    comp_out_dir.mkdir(exist_ok=True, parents=True)

    # 1. PR Classification Metrics Comparison
    class_metrics_for_comp_dict = {model_name: res_data['pr_classification_metrics'] 
                                   for model_name, res_data in results_per_pr_model.items() 
                                   if 'pr_classification_metrics' in res_data}
    if class_metrics_for_comp_dict:
        df_class_comp = pd.DataFrame(class_metrics_for_comp_dict).T # Models as rows, metrics as columns
        df_class_comp.to_csv(comp_out_dir / 'pr_classification_metrics_comparison.csv')
        print(f"\\nPR Classification Metrics Comparison for {tissue_type_to_test.upper()}:")
        print(df_class_comp)
        try:
            df_class_comp.plot(kind='bar', figsize=(12,7), rot=0)
            plt.title(f'PR Classification Metrics Comparison ({tissue_type_to_test.upper()})')
            plt.ylabel('Score')
            plt.xlabel('PR Model')
            plt.legend(title='Metric', bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.tight_layout()
            plt.savefig(comp_out_dir / 'pr_classification_metrics_comparison.png', dpi=300)
            plt.close()
        except Exception as e:
            print(f"Error plotting PR classification metrics comparison: {e}")
    else:
        print("No PR classification metrics available to compare for PR models.")

    # 2. Average Direct MM SSIM Comparison (A_vs_B, A_vs_C, B_vs_C averages)
    avg_mm_ssim_for_comp_dict = {}
    for model_name, res_data in results_per_pr_model.items():
        if 'avg_direct_mm_ssim_summary' in res_data:
            # res_data['avg_direct_mm_ssim_summary'] is DataFrame: Index=M_ij, Cols=['A_vs_B', 'A_vs_C', 'B_vs_C']
            summary_df = res_data['avg_direct_mm_ssim_summary']
            if not summary_df.empty:
                avg_mm_ssim_for_comp_dict[model_name] = {
                    'A_vs_B_MM_SSIM_AvgElements': summary_df['A_vs_B'].mean(skipna=True),
                    'A_vs_C_MM_SSIM_AvgElements': summary_df['A_vs_C'].mean(skipna=True),
                    'B_vs_C_MM_SSIM_AvgElements': summary_df['B_vs_C'].mean(skipna=True)
                }
    if avg_mm_ssim_for_comp_dict:
        df_avg_mm_ssim_comp = pd.DataFrame(avg_mm_ssim_for_comp_dict).T # Models as rows
        df_avg_mm_ssim_comp.to_csv(comp_out_dir / 'avg_direct_mm_ssim_comparison.csv')
        print(f"\\nAvg. Direct MM SSIM Comparison for {tissue_type_to_test.upper()} (across PR Models):")
        print(df_avg_mm_ssim_comp)
        try:
            df_avg_mm_ssim_comp.plot(kind='bar', figsize=(10,6), rot=0)
            plt.title(f'Avg. Direct MM SSIM Comparison - PR Models ({tissue_type_to_test.upper()})')
            plt.ylabel('Avg. SSIM over MM elements')
            plt.xlabel('PR Model')
            plt.legend(title='MM Comparison Type')
            plt.tight_layout()
            plt.savefig(comp_out_dir / 'avg_direct_mm_ssim_comparison.png', dpi=300)
            plt.close()
        except Exception as e:
            print(f"Error plotting avg direct MM SSIM comparison: {e}")
    else:
        print("No average direct MM SSIM data available to compare for PR models.")
        
    return results_per_pr_model # Return all collected eval data for potential further use


In [8]:
def run_notebook_demonstration(tissue_type='brain', pr_model='xgb', 
                               visualize_mm_maps=True, 
                               run_decomposition_analysis=True, 
                               ml_decomp_model_filename='decomposition_cnn.keras'):
    """
    Demonstration function for a single tissue type and PR model.
    Includes evaluation and optional decomposition analysis.
    """
    print(f"\\n{'*'*20} NOTEBOOK DEMO: {tissue_type.upper()} (PR Model: {pr_model.upper()}) {'*'*20}")
    
    sample_sets_available = get_sample_sets()
    if tissue_type not in sample_sets_available:
        print(f"Invalid tissue type for demo: {tissue_type}. Available: {list(sample_sets_available.keys())}")
        return None
    
    # 1. Run Evaluation
    # `visualize_first_n` in `evaluate_tissue_samples` controls MM map visualization
    eval_results = evaluate_tissue_samples(
        tissue_type, 
        sample_sets_available[tissue_type], 
        pr_model_name=pr_model,
        visualize_first_n=1 if visualize_mm_maps else 0 
    )
    
    if not eval_results: # Check if evaluation itself failed
        print(f"Demo evaluation failed for {tissue_type} with PR model {pr_model}.")
        return None

    print("\\nDemo Evaluation Summary:")
    if 'pr_classification_metrics' in eval_results:
        print(f"  PR Classification Metrics: {eval_results['pr_classification_metrics']}")
    
    if 'avg_direct_mm_ssim_summary' in eval_results and not eval_results['avg_direct_mm_ssim_summary'].empty:
        print("  Average Direct MM SSIM (element-wise means):")
        # Print the mean of each comparison column (A_vs_B, etc.)
        print(eval_results['avg_direct_mm_ssim_summary'].mean()) 
    
    # 2. Run Decomposition Analysis (Optional)
    decomposition_results = None
    if run_decomposition_analysis:
        print(f"\\n--- Demo: Running Lu-Chipman Decomposition for {tissue_type.upper()} (PR Model: {pr_model.upper()}) ---")
        # analyze_lu_chipman_decomposition expects `results_eval` which is `eval_results` here.
        # It also needs `tissue_type` and the `pr_model_type_tag`.
        decomposition_results = analyze_lu_chipman_decomposition(
            results_eval=eval_results, # This is the dict output from evaluate_tissue_samples
            tissue_type=tissue_type,
            pr_model_type_tag=pr_model, # For file naming within the util
            ml_decomp_model_name=ml_decomp_model_filename # Pass ML model name for decomp
        )
        
        if decomposition_results:
            print("\\nDemo Decomposition Analysis Highlights:")
            if 'gt_pipeline_summary_df' in decomposition_results and not decomposition_results['gt_pipeline_summary_df'].empty:
                print("  GT Decomp. Pipeline - Avg SSIM for Optical Params:")
                print(decomposition_results['gt_pipeline_summary_df'])
            if 'ml_pipeline_summary_df' in decomposition_results and not decomposition_results['ml_pipeline_summary_df'].empty:
                print("\\n  ML Decomp. Pipeline - Avg SSIM for Optical Params (vs A_gt):")
                print(decomposition_results['ml_pipeline_summary_df'])
            print(f"Decomposition plots and detailed CSVs have been saved in the results directory for {tissue_type}.")
        else:
            print("Demo decomposition analysis did not produce results or was skipped.")
            
    print(f"\\n{'*'*20} DEMO COMPLETED for {tissue_type.upper()} (PR Model: {pr_model.upper()}) {'*'*20}")
    # Return both evaluation and decomposition results for inspection if needed
    return {'evaluation': eval_results, 'decomposition': decomposition_results}


## Execution Workflows

In [9]:
# The cells below provide examples of how to run the analyses.
# You can uncomment and adapt them as needed.
# Make sure the `testing_utils` directory is in the same parent directory as this notebook,
# or adjust `sys.path` accordingly at the top of the notebook.

### Workflow 1: Full Evaluation & Decomposition for a chosen PR model

In [9]:
# This workflow runs evaluations for all available tissue types using a specified 
# PR model. Then, it performs Lu-Chipman decomposition analysis on these results,
# generating both GT-based and ML-based (if model provided) decomposition parameter comparisons.
# --- Configuration for Workflow 1 ---
WF1_PR_MODEL_CHOICE = 'xgb'  # Options: 'xgb', 'mlp', 'cat'
WF1_ML_DECOMP_MODEL_FILENAME = 'decomposition_cnn.keras'  # Filename in models dir, or None to skip ML decomposition
WF1_VISUALIZE_MM_FOR_N_SAMPLES = 1  # Visualize MMs for the first N samples per tissue type (0 for none)

In [10]:
# # --- Step 1: Run evaluations for all tissues using the chosen PR model ---
# print(f"WORKFLOW 1: Starting full evaluations for PR Model: {WF1_PR_MODEL_CHOICE.upper()}")
# all_eval_results = run_all_tissue_evaluations(
#     pr_model_to_use=WF1_PR_MODEL_CHOICE,
#     visualize_first_n_samples_per_tissue=WF1_VISUALIZE_MM_FOR_N_SAMPLES
# )
# # `all_eval_results` is a dict: {tissue_name: evaluation_output_dict, ...}
# # `create_overall_summary` is called within `run_all_tissue_evaluations` to save summaries.
# 

In [11]:
# # --- Step 2: Run Lu-Chipman decomposition analysis for each successfully evaluated tissue ---
# all_decomposition_results = {}
# if all_eval_results: # Check if there are any results from the evaluation step
#     print(f"\\nWORKFLOW 1: Starting Lu-Chipman decomposition analysis based on {WF1_PR_MODEL_CHOICE.upper()} PR model results.")
#     for tissue_name, individual_tissue_eval_data in all_eval_results.items():
#         if individual_tissue_eval_data: # Ensure eval data exists for this tissue
#             print(f"\\n--- Analyzing decomposition for tissue: {tissue_name.upper()} ---")
#             # `analyze_lu_chipman_decomposition` takes the output of `evaluate_tissue_samples`
#             decomp_output_for_tissue = analyze_lu_chipman_decomposition(
#                 results_eval=individual_tissue_eval_data, 
#                 tissue_type=tissue_name,
#                 pr_model_type_tag=WF1_PR_MODEL_CHOICE, # For file naming consistency
#                 ml_decomp_model_name=WF1_ML_DECOMP_MODEL_FILENAME
#             )
#             if decomp_output_for_tissue:
#                 all_decomposition_results[tissue_name] = decomp_output_for_tissue
#         else:
#             print(f"Skipping decomposition for {tissue_name.upper()} as its evaluation data is missing or evaluation failed.")
#     print("\\nWORKFLOW 1: Lu-Chipman decomposition analysis finished for all successfully evaluated tissues.")
# else:
#     print("WORKFLOW 1: Evaluation step failed or produced no results for any tissue. Skipping all decomposition analyses.")
# 

In [13]:
# # --- Step 3: (Optional) Further combined summary of decomposition results ---
# # The `analyze_lu_chipman_decomposition` utility saves detailed CSVs and summary plots per tissue.
# # If you need a grand summary table/plot combining all tissues' decomposition SSIMs,
# # you would iterate through `all_decomposition_results` and aggregate the 'gt_pipeline_summary_df' 
# # and 'ml_pipeline_summary_df' DataFrames. This might involve another helper function.
# # For now, you can access individual summaries like this:
# if 'brain' in all_decomposition_results and all_decomposition_results.get('brain'): # Check if 'brain' key exists and its value is not None
#     brain_decomp_data = all_decomposition_results['brain']
#     brain_gt_decomp_summary = brain_decomp_data.get('gt_pipeline_summary_df')
#     brain_ml_decomp_summary = brain_decomp_data.get('ml_pipeline_summary_df')
#     print("\\nExample - Brain GT Decomposition Summary (from Workflow 1):")
#     if brain_gt_decomp_summary is not None: 
#         print(brain_gt_decomp_summary)
#     else:
#         print("Brain GT Decomposition Summary not available.")
#     if brain_ml_decomp_summary is not None:
#         print("\\nExample - Brain ML Decomposition Summary (from Workflow 1):")
#         print(brain_ml_decomp_summary)
#     else:
#         print("Brain ML Decomposition Summary not available (ML model might not have been run or no results).")
# 
# print(f"\\nWORKFLOW 1 (using PR Model: {WF1_PR_MODEL_CHOICE.upper()}) COMPLETED.")

### Workflow 2: Compare different PR Models on a single Tissue Type

In [14]:
# This workflow focuses on evaluating how different PR models ('mlp', 'xgb', 'cat')
# perform on a specific tissue type (e.g., 'brain'). 
# It generates comparative plots for PR classification metrics and average direct MM SSIMs.
# Decomposition analysis is not part of this specific workflow but could be added as a sub-step
# for the "best" performing PR model, or for all of them if desired.
# --- Configuration for Workflow 2 ---
#WF2_TARGET_TISSUE_FOR_PR_COMPARISON = 'brain'  # Choose tissue: 'brain', 'cervix', or 'afmmm'
# WF2_PR_MODELS_TO_EVALUATE = ['mlp', 'xgb', 'cat'] # List of PR models to compare

# --- Run PR Model Comparison ---
# print(f"\\nWORKFLOW 2: Comparing PR Models {WF2_PR_MODELS_TO_EVALUATE} on tissue: {WF2_TARGET_TISSUE_FOR_PR_COMPARISON.upper()}")
# comparison_eval_data = compare_pr_models_on_tissue(
#     tissue_type_to_test=WF2_TARGET_TISSUE_FOR_PR_COMPARISON,
#     pr_models_list=WF2_PR_MODELS_TO_EVALUATE
# )
# # `comparison_eval_data` is a dict: {pr_model_name: evaluation_output_dict, ...}
# # The function `compare_pr_models_on_tissue` saves its own summary CSVs and plots.

# if comparison_eval_data:
#    print(f"\\nPR Model comparison for {WF2_TARGET_TISSUE_FOR_PR_COMPARISON.upper()} complete. Results are saved in the 'pr_model_comparison' directory.")
#    # You can further inspect the contents of `comparison_eval_data` here if needed.
#    # For example, to get the XGB results for brain:
#    # xgb_brain_results_from_comparison = comparison_eval_data.get('xgb')
#    # if xgb_brain_results_from_comparison:
#    #     print("\\nXGB Brain evaluation data from comparison run:")
#    #     print(xgb_brain_results_from_comparison['pr_classification_metrics'])
# else:
#    print(f"\\nPR Model comparison for {WF2_TARGET_TISSUE_FOR_PR_COMPARISON.upper()} failed or produced no results.")

# print(f"\\nWORKFLOW 2 COMPLETED.")

### Workflow 3: Run a Quick Demonstration for a single case

In [15]:
# # This uses the `run_notebook_demonstration` function for a quick check of 
# # evaluation and (optionally) decomposition for one tissue and one PR model.
# # --- Configuration for Workflow 3 ---
# WF3_DEMO_TISSUE_TYPE = 'cervix'
# WF3_DEMO_PR_MODEL_TYPE = 'xgb'
# WF3_DEMO_VISUALIZE_MM_MAPS = True       # Visualize MMs for the first sample
# WF3_DEMO_RUN_DECOMPOSITION = True       # Run Lu-Chipman analysis after evaluation
# WF3_DEMO_ML_DECOMP_MODEL_FILE = 'decomposition_cnn.keras' # Filename or None
# 
# # --- Run Demonstration ---
# print(f"\\nWORKFLOW 3: Running demonstration for {WF3_DEMO_TISSUE_TYPE.upper()} (PR Model: {WF3_DEMO_PR_MODEL_TYPE.upper()})")
# demo_run_results = run_notebook_demonstration(
#     tissue_type=WF3_DEMO_TISSUE_TYPE,
#     pr_model=WF3_DEMO_PR_MODEL_TYPE,
#     visualize_mm_maps=WF3_DEMO_VISUALIZE_MM_MAPS,
#     run_decomposition_analysis=WF3_DEMO_RUN_DECOMPOSITION,
#     ml_decomp_model_filename=WF3_DEMO_ML_DECOMP_MODEL_FILE
# )
# 
# if demo_run_results and demo_run_results.get('evaluation'):
#     print("\\nDemonstration finished successfully. Evaluation results are available in 'demo_run_results['evaluation']'.")
#     if demo_run_results.get('decomposition'):
#         print("Decomposition analysis results are available in 'demo_run_results['decomposition']'.")
# else:
#     print("\\nDemonstration encountered an issue or evaluation failed.")
# 
# print(f"\\nWORKFLOW 3 COMPLETED.")

In [17]:
 # This workflow specifically compares the ML filtering with GT filtering (Method D)
# and calculates both traditional classification metrics and SSIM between the filtered data.
# --- Configuration for Workflow 4 ---
WF4_TARGET_TISSUE = 'brain'  # Choose tissue type to analyze
WF4_PR_MODEL = 'xgb'  # PR model to use: 'xgb', 'mlp', or 'cat'
WF4_VISUALIZE_ALL_SAMPLES = False  # Set to True to visualize all samples, False for only first sample

In [23]:
# Final corrected version of run_filtering_comparison

def run_filtering_comparison(tissue_type, pr_model='xgb', visualize_all=False):
    """
    Run comparison between GT filtering and ML filtering, including SSIM metrics.
    
    Args:
        tissue_type: Tissue type to analyze
        pr_model: PR model to use for ML filtering
        visualize_all: Whether to visualize all samples
        
    Returns:
        Dictionary with comparison results
    """
    print(f"\n{'*'*20} WORKFLOW 4: FILTERING COMPARISON (GT vs ML) {'*'*20}")
    print(f"Tissue Type: {tissue_type.upper()}, PR Model: {pr_model.upper()}")
    
    # 1. Load data
    sample_sets = get_sample_sets()
    if tissue_type not in sample_sets:
        print(f"Invalid tissue type: {tissue_type}. Available: {list(sample_sets.keys())}")
        return None
    
    # 2. Load sample files
    arrays, sample_names, dims = load_sample_files(tissue_type, sample_sets[tissue_type])
    if not arrays:
        print(f"No data loaded for {tissue_type}. Aborting comparison.")
        return None
    
    # 3. Prepare data
    X_flat, y_flat_gt_pr, iso_merged = merge_and_prepare_data(arrays)
    if X_flat is None:
        print(f"Data preparation failed for {tissue_type}. Aborting comparison.")
        return None
    
    # 4. Get ML PR predictions
    y_pred_ml_pr, probs_ml_pr = load_pr_model_and_predict(X_flat, use_model=pr_model)
    
    # 5. Reshape predictions to match sample dimensions
    num_samples, H, W, _ = iso_merged.shape
    ml_pr_mask_per_sample = y_pred_ml_pr.reshape(num_samples, H, W)
    gt_pr_mask_per_sample = y_flat_gt_pr.reshape(num_samples, H, W)
    
    # 6. Run comparison
    # Only visualize first sample to avoid excessive output, unless specifically requested
    visualize_samples = len(arrays) if visualize_all else 1
    arrays_to_visualize = arrays[:visualize_samples]
    sample_names_to_visualize = sample_names[:visualize_samples]
    ml_mask_to_visualize = ml_pr_mask_per_sample[:visualize_samples]
    gt_mask_to_visualize = gt_pr_mask_per_sample[:visualize_samples]
    
    # Calculate the number of pixels in the selected samples
    n_pix = H * W
    total_pixels = visualize_samples * n_pix
    
    # Only use the probabilities corresponding to the visualized samples
    probs_subset = probs_ml_pr[:total_pixels]
    
    comparison_results = compare_filtering_methods(
        arrays_to_visualize,
        sample_names_to_visualize,
        ml_mask_to_visualize,
        gt_mask_to_visualize,
        probs_subset,  # Use only the relevant subset of probabilities
        dims
    )
    
    # 7. Display results
    print("\nOverall Classification Metrics:")
    for metric, value in comparison_results['overall_metrics'].items():
        print(f"  {metric.capitalize()}: {value:.4f}")
    
    print("\nAverage SSIM Between GT and ML Filtering by Mueller Matrix Element:")
    for element, ssim_val in comparison_results['avg_ssim'].items():
        print(f"  {element}: {ssim_val:.4f}")
    
    print(f"\nDetailed per-sample SSIM values saved to: {file_paths.results}/filtering_comparison/")
    print(f"Visualization of the filtered data saved to: {file_paths.figures}/filtering_comparison/")
    
    # Create additional visualizations for the filtering comparison
    # ROC Curves and PR curves have already been created in compare_filtering_methods function
    
    # Create customized report for filtering comparison
    filtering_report_dir = file_paths.results / 'filtering_comparison' / f'{tissue_type}_{pr_model}'
    filtering_report_dir.mkdir(exist_ok=True, parents=True)
    
    # Create a comprehensive visual report combining metrics and SSIM
    if comparison_results:
        overall_metrics = comparison_results.get('overall_metrics', {})
        avg_ssim = comparison_results.get('avg_ssim', {})
        
        # Create a 4-panel visualization comparing various metrics
        fig, axs = plt.subplots(2, 2, figsize=(16, 12))
        
        # Panel 1: Classification Metrics
        metrics_names = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
        metrics_values = [overall_metrics.get(m, 0) for m in metrics_names]
        axs[0, 0].bar(metrics_names, metrics_values, color='skyblue')
        axs[0, 0].set_title('Classification Metrics', fontsize=14, fontweight='bold')
        axs[0, 0].set_ylim(0, 1.05)
        axs[0, 0].set_ylabel('Score')
        axs[0, 0].tick_params(axis='x', rotation=45)
        for i, v in enumerate(metrics_values):
            axs[0, 0].text(i, v + 0.03, f'{v:.3f}', ha='center')
        
        # Panel 2: SSIM by Element
        if avg_ssim:
            elements = sorted(avg_ssim.keys())
            ssim_values = [avg_ssim[k] for k in elements]
            axs[0, 1].bar(elements, ssim_values, color='lightgreen')
            axs[0, 1].set_title('SSIM between GT and ML Filtering', fontsize=14, fontweight='bold')
            axs[0, 1].set_ylim(0, 1.05)
            axs[0, 1].set_ylabel('SSIM')
            axs[0, 1].tick_params(axis='x', rotation=45)
            for i, v in enumerate(ssim_values):
                axs[0, 1].text(i, v + 0.03, f'{v:.3f}', ha='center')
        
        # Panel 3: PR Distribution visualization
        # Create distribution of predicted PR vs. actual PR
        sample_results = comparison_results.get('sample_results', [])
        if sample_results:
            gt_counts = [s.get('gt_pr_count', 0) for s in sample_results]
            ml_counts = [s.get('ml_pr_count', 0) for s in sample_results]
            sample_labels = [s.get('sample_name', f"Sample {i}") for i, s in enumerate(sample_results)]
            
            x = np.arange(len(sample_labels))
            width = 0.35
            
            axs[1, 0].bar(x - width/2, gt_counts, width, label='GT PR', color='cornflowerblue')
            axs[1, 0].bar(x + width/2, ml_counts, width, label='ML PR', color='lightcoral')
            axs[1, 0].set_title('PR Pixel Counts by Sample', fontsize=14, fontweight='bold')
            axs[1, 0].set_ylabel('Pixel Count')
            axs[1, 0].set_xticks(x)
            axs[1, 0].set_xticklabels(sample_labels, rotation=45, ha='right')
            axs[1, 0].legend()
            
            # Add percentage difference labels
            for i, (gt, ml) in enumerate(zip(gt_counts, ml_counts)):
                if gt > 0:
                    pct_diff = (ml - gt) / gt * 100
                    axs[1, 0].text(i, max(gt, ml) + 50, f"{pct_diff:.1f}%", ha='center')
        
        # Panel 4: Load and display the ROC curve here
        # The ROC curve is already generated, so we'll load and display it
        roc_path = file_paths.results / 'filtering_comparison' / 'roc_curve_overall.png'
        if roc_path.exists():
            try:
                img = plt.imread(str(roc_path))
                axs[1, 1].imshow(img)
                axs[1, 1].axis('off')
            except Exception as e:
                print(f"Could not load ROC curve image: {e}")
                axs[1, 1].text(0.5, 0.5, 'ROC Curve not available', ha='center', va='center', fontsize=14)
                axs[1, 1].axis('off')
        else:
            axs[1, 1].text(0.5, 0.5, 'ROC Curve not available', ha='center', va='center', fontsize=14)
            axs[1, 1].axis('off')
        
        plt.suptitle(f'Method D: Filtering Comparison - {tissue_type.upper()} (PR Model: {pr_model.upper()})',
                   fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.subplots_adjust(top=0.95)
        
        try:
            plt.savefig(filtering_report_dir / 'comprehensive_metrics_report.png', dpi=300, bbox_inches='tight')
        except Exception as e:
            print(f"Could not save comprehensive report: {e}")
        
        plt.close()
    
    return comparison_results

In [24]:
# Run the filtering comparison workflow
filtering_comparison_results = run_filtering_comparison(
    tissue_type=WF4_TARGET_TISSUE,
    pr_model=WF4_PR_MODEL,
    visualize_all=WF4_VISUALIZE_ALL_SAMPLES
)

if filtering_comparison_results:
    print("\nFiltering comparison completed successfully.")
else:
    print("\nFiltering comparison workflow encountered issues or produced no results.")

print(f"\nWORKFLOW 4 COMPLETED.")


******************** WORKFLOW 4: FILTERING COMPARISON (GT vs ML) ********************
Tissue Type: BRAIN, PR Model: XGB
Found 5 brain files for evaluation.


Loading brain samples: 100%|██████████| 5/5 [00:00<00:00, 200.10it/s]

Merged shape: (5, 388, 516, 17)


Using XGBoost model for PR prediction

Overall ROC AUC: 0.9965
Overall PR AUC: 0.9998
Saved PDF successfully to /Users/chaechae/Desktop/EP_Code/pr_prediction/results/figures/filtering_comparison/Method_D_Mueller_2023-03-22_T_HORAO-91-BF_FR_S1_1_GT_filtering_550.pdf
Saved PDF successfully to /Users/chaechae/Desktop/EP_Code/pr_prediction/results/figures/filtering_comparison/Method_D_Mueller_2023-03-22_T_HORAO-91-BF_FR_S1_1_ML_filtering_550.pdf

Overall Classification Metrics:
  Accuracy: 0.9873
  Precision: 0.9908
  Recall: 0.9960
  F1: 0.9934
  Roc_auc: 0.9965

Average SSIM Between GT and ML Filtering by Mueller Matrix Element:
  M11: 0.9430
  M12: 0.9890
  M13: 0.9945
  M14: 0.9980
  M21: 0.9892
  M22: 0.9503
  M23: 0.9706
  M24: 0.9837
  M31: 0.9857
  M32: 0.9836
  M33: 0.9488
  M34: 0.9881

Detailed per-sample SSIM values saved to: /Users/chaechae/Desktop/EP_Code/pr_prediction/results/filtering_comparison/
Visualization of the filtered data saved to: /Users/chaechae/Desktop/EP_Code/p

AttributeError: module 'matplotlib' has no attribute 'subplots'

### Workflow 5: Generate Integrated Comparison Report (A, B, C, D Methods)

In [12]:
# This workflow creates a comprehensive report that integrates metrics from all methods:
# - Method A: GT filtering, GT last row, GT decomposition
# - Method B: GT filtering, ML last row, GT decomposition
# - Method C: ML filtering, ML last row, GT decomposition
# - Method D: Direct comparison of GT filtering vs ML filtering with SSIM
#
# The integrated report helps visualize and compare the performance across all methods.

In [ ]:
def generate_integrated_report(tissue_type='brain', pr_model='xgb'):
    """
    Generate an integrated report comparing all methods (A, B, C, D).

    Args:
        tissue_type: Tissue type to analyze
        pr_model: PR model to use

    Returns:
        Dictionary with combined results
    """
    print(f"\n{'*'*20} WORKFLOW 5: INTEGRATED COMPARISON REPORT {'*'*20}")
    print(f"Tissue Type: {tissue_type.upper()}, PR Model: {pr_model.upper()}")

    # 1. Run evaluation for methods A, B, C
    print("\nRunning evaluation for Methods A, B, C...")
    abc_results = evaluate_tissue_samples(
        tissue_type=tissue_type,
        sample_filenames_set=get_sample_sets()[tissue_type],
        pr_model_name=pr_model,
        visualize_first_n=1  # Visualize only first sample
    )

    # 2. Run comparison for Method D (GT vs ML filtering)
    print("\nRunning comparison for Method D (GT vs ML filtering)...")
    d_results = run_filtering_comparison(
        tissue_type=tissue_type,
        pr_model=pr_model,
        visualize_all=False  # Visualize only first sample
    )

    if not abc_results or not d_results:
        print("One or more evaluation methods failed. Cannot generate integrated report.")
        return None

    # 3. Combine results into integrated report
    integrated_results = {
        'tissue_type': tissue_type,
        'pr_model': pr_model,
        'abc_methods': abc_results,
        'filtering_method_d': d_results
    }

    # 4. Generate visualizations comparing all methods
    print("\nGenerating integrated visualizations...")

    # Output directory for integrated report
    report_dir = file_paths.results / tissue_type / 'integrated_report'
    report_dir.mkdir(exist_ok=True, parents=True)

    # Create ROC and PR curves for the integrated report
    if d_results:
        # Get ground truth and ML predictions for ROC curve from filtering method
        samples_data = get_sample_sets()[tissue_type]
        arrays, sample_names, dims = load_sample_files(tissue_type, samples_data)

        if arrays:
            X_flat, y_flat_gt_pr, _ = merge_and_prepare_data(arrays)
            if X_flat is not None and y_flat_gt_pr is not None:
                _, probs_ml_pr = load_pr_model_and_predict(X_flat, use_model=pr_model)

                # Plot ROC and PR curves for the integrated report
                plot_roc_curve(y_flat_gt_pr, probs_ml_pr, report_dir, "integrated")
                plot_precision_recall_curve(y_flat_gt_pr, probs_ml_pr, report_dir, "integrated")

                # Create confusion matrix visualization
                try:
                    from sklearn.metrics import confusion_matrix
                    import seaborn as sns

                    y_pred_ml_pr = (probs_ml_pr > 0.5).astype(int)
                    cm = confusion_matrix(y_flat_gt_pr, y_pred_ml_pr)

                    plt.figure(figsize=(8, 6))
                    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                                xticklabels=['Non-PR', 'PR'],
                                yticklabels=['Non-PR', 'PR'])
                    plt.xlabel('Predicted')
                    plt.ylabel('True')
                    plt.title(f'Confusion Matrix - {tissue_type.upper()} (PR Model: {pr_model.upper()})')
                    plt.tight_layout()
                    plt.savefig(report_dir / 'confusion_matrix.png', dpi=300, bbox_inches='tight')
                    plt.close()
                except Exception as e:
                    print(f"Could not create confusion matrix: {e}")

    # 4.1. Create detailed metrics comparison tables
    # PR Classification metrics from Method D
    filtering_metrics = d_results.get('overall_metrics', {})

    # For Method A, B, C - Direct MM SSIM
    direct_mm_ssim_df = abc_results.get('avg_direct_mm_ssim_summary')

    # Filtering SSIM metrics from Method D
    filtering_avg_ssim = d_results.get('avg_ssim', {})

    # PR classification metrics
    pr_metrics = abc_results.get('pr_classification_metrics', {})

    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    # Combined metrics visualization
    plt.figure(figsize=(15, 12))

    # Create a 2x2 subplot layout
    gs = plt.GridSpec(2, 2, height_ratios=[1, 1.5])

    # Plot 1: PR Classification Metrics (Method D)
    ax1 = plt.subplot(gs[0, 0])
    if pr_metrics:
        metrics_df = pd.DataFrame([pr_metrics])
        metrics_df = metrics_df[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']]
        metrics_df.plot(kind='bar', ax=ax1, rot=0, color='cornflowerblue')
        ax1.set_title('PR Classification Metrics', fontsize=14, fontweight='bold')
        ax1.set_ylim(0, 1)
        ax1.set_ylabel('Score')
        for i, v in enumerate(metrics_df.values[0]):
            ax1.text(i, v + 0.03, f'{v:.3f}', ha='center', fontsize=10)
        ax1.get_legend().remove()  # Remove legend as it's a single row

    # Plot 2: Average Direct MM SSIM (Methods A, B, C)
    ax2 = plt.subplot(gs[0, 1])
    if direct_mm_ssim_df is not None and not direct_mm_ssim_df.empty:
        # Calculate mean across MM elements for each comparison
        avg_ssim_values = direct_mm_ssim_df.mean()
        colors = ['green', 'orange', 'purple']
        ax2.bar(avg_ssim_values.index, avg_ssim_values.values, color=colors)
        ax2.set_title('Avg. Direct MM SSIM - Methods A, B, C', fontsize=14, fontweight='bold')
        ax2.set_ylim(0, 1)
        ax2.set_ylabel('Avg. SSIM')
        for i, v in enumerate(avg_ssim_values.values):
            ax2.text(i, v + 0.03, f'{v:.3f}', ha='center', fontsize=10)

    # Plot 3: Filtering Metrics Detail - GT vs ML
    ax3 = plt.subplot(gs[1, 0])
    if filtering_metrics:
        metrics_names = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
        values = [filtering_metrics.get(m, 0) for m in metrics_names]

        # Create a comparison bar chart
        ax3.bar(metrics_names, values, color='darkorange', alpha=0.8)
        ax3.set_title('Method D: Filtering Metrics Detail', fontsize=14, fontweight='bold')
        ax3.set_ylim(0, 1)
        ax3.set_ylabel('Score')
        ax3.set_xticklabels(metrics_names, rotation=45, ha='right')

        # Add value labels above bars
        for i, v in enumerate(values):
            ax3.text(i, v + 0.03, f'{v:.3f}', ha='center', fontsize=10)

    # Plot 4: SSIM by Mueller Element for Method D
    ax4 = plt.subplot(gs[1, 1])
    if filtering_avg_ssim:
        elements = list(filtering_avg_ssim.keys())
        ssim_values = list(filtering_avg_ssim.values())

        # Sort by element name to get a logical order
        sorted_indices = sorted(range(len(elements)), key=lambda i: elements[i])
        sorted_elements = [elements[i] for i in sorted_indices]
        sorted_values = [ssim_values[i] for i in sorted_indices]

        # Create a bar chart with gradient color
        bars = ax4.bar(sorted_elements, sorted_values, color=plt.cm.viridis(np.linspace(0.2, 0.8, len(sorted_elements))))
        ax4.set_title('Method D: SSIM by Mueller Element', fontsize=14, fontweight='bold')
        ax4.set_ylim(0, 1)
        ax4.set_ylabel('SSIM')
        ax4.set_xticklabels(sorted_elements, rotation=45, ha='right')

        # Add value labels above bars
        for i, v in enumerate(sorted_values):
            ax4.text(i, v + 0.03, f'{v:.3f}', ha='center', fontsize=9)

    plt.suptitle(f'Integrated Analysis: {tissue_type.upper()} (PR Model: {pr_model.upper()})', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.savefig(report_dir / f'integrated_metrics_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()

    # 4.2. Create consolidated metrics table with method descriptions
    # Create a description of the methods
    method_descriptions = {
        'Method A': 'GT filtering + GT last row + GT decomposition',
        'Method B': 'GT filtering + ML last row + GT decomposition',
        'Method C': 'ML filtering + ML last row + GT decomposition',
        'Method D': 'Direct comparison of GT filtering vs ML filtering'
    }

    # Combine PR classification metrics with filtering metrics in a table
    combined_metrics = {}
    if pr_metrics:
        combined_metrics.update({f"PR_{k}": v for k, v in pr_metrics.items()})
    if filtering_metrics:
        combined_metrics.update({f"Filtering_{k}": v for k, v in filtering_metrics.items()})

    # Add average SSIM from direct comparisons
    if direct_mm_ssim_df is not None and not direct_mm_ssim_df.empty:
        avg_ssim_values = direct_mm_ssim_df.mean()
        combined_metrics.update({f"Avg_{k}_SSIM": v for k, v in avg_ssim_values.items()})

    # Create and save consolidated metrics table
    metrics_df = pd.DataFrame([combined_metrics])
    metrics_df.to_csv(report_dir / f'consolidated_metrics.csv', index=False)

    # Create method description table
    methods_df = pd.DataFrame.from_dict(method_descriptions, orient='index', columns=['Description'])
    methods_df.index.name = 'Method'
    methods_df.to_csv(report_dir / f'method_descriptions.csv')

    # Generate a comprehensive summary report in text format
    with open(report_dir / f'summary_report.txt', 'w') as f:
        f.write(f"INTEGRATED ANALYSIS REPORT\n")
        f.write(f"=========================\n")
        f.write(f"Tissue Type: {tissue_type.upper()}\n")
        f.write(f"PR Model: {pr_model.upper()}\n\n")

        f.write("METHOD DESCRIPTIONS\n")
        f.write("-----------------\n")
        for method, desc in method_descriptions.items():
            f.write(f"{method}: {desc}\n")

        f.write("\nPR CLASSIFICATION METRICS\n")
        f.write("------------------------\n")
        if pr_metrics:
            for metric, value in pr_metrics.items():
                f.write(f"{metric.capitalize()}: {value:.4f}\n")

        f.write("\nFILTERING COMPARISON (Method D)\n")
        f.write("-----------------------------\n")
        if filtering_metrics:
            for metric, value in filtering_metrics.items():
                f.write(f"{metric.capitalize()}: {value:.4f}\n")

        f.write("\nAVERAGE DIRECT MM SSIM (Methods A, B, C)\n")
        f.write("-------------------------------------\n")
        if direct_mm_ssim_df is not None and not direct_mm_ssim_df.empty:
            avg_ssim_values = direct_mm_ssim_df.mean()
            for comp, value in avg_ssim_values.items():
                f.write(f"{comp}: {value:.4f}\n")

        f.write("\nAVERAGE SSIM BY MUELLER ELEMENT (Method D)\n")
        f.write("----------------------------------------\n")
        if filtering_avg_ssim:
            elements = sorted(filtering_avg_ssim.keys())
            for element in elements:
                f.write(f"{element}: {filtering_avg_ssim[element]:.4f}\n")

        f.write("\nThis report was generated automatically by the integrated analysis workflow.\n")

    print(f"Integrated report saved to: {report_dir}")
    return integrated_results

In [ ]:
# Run the integrated report workflow
integrated_report_results = generate_integrated_report(
    tissue_type='brain',
    pr_model='xgb'
)

if integrated_report_results:
    print("\nIntegrated report generated successfully!")
else:
    print("\nIntegrated report generation encountered issues.")

print(f"\nWORKFLOW 5 COMPLETED.")